In [1]:
import duckdb, json, os, argparse


def main(parquet_dir, rules_dir, output_dir):
    """Run the (frozen) rules in rules.json against the parquet and write per-rule
    failure counts to results.json. parquet_dir/rules_dir/output_dir are folder paths."""
    parquet = os.path.join(parquet_dir, "dataset.parquet")
    with open(os.path.join(rules_dir, "rules.json")) as f:
        rules = json.load(f)["rules"]

    con = duckdb.connect()
    con.execute("CREATE OR REPLACE VIEW dataset AS "
                "SELECT * FROM read_parquet('%s')" % parquet.replace("\\", "/"))
    n_rows = con.execute("SELECT count(*) FROM dataset").fetchone()[0]

    row_rules = [r for r in rules if r["result_kind"] == "row"]
    agg_rules = [r for r in rules if r["result_kind"] == "aggregate"]
    res = {}

    # all per-row checks in one scan: failed_count = rows not satisfying the rule
    if row_rules:
        sel = ", ".join('count_if(NOT (%s)) AS c%d' % (r["sql"], i)
                        for i, r in enumerate(row_rules))
        for r, v in zip(row_rules, con.execute("SELECT %s FROM dataset" % sel).fetchone()):
            res[r["id"]] = {"result_kind": "row", "failed_count": int(v)}

    # all dataset-level checks in one pass: sql is the PASS condition -> 0 if pass, 1 if fail
    if agg_rules:
        sel = ", ".join('(%s) AS a%d' % (r["sql"], i) for i, r in enumerate(agg_rules))
        for r, v in zip(agg_rules, con.execute("SELECT %s FROM dataset" % sel).fetchone()):
            res[r["id"]] = {"result_kind": "aggregate", "failed_count": 0 if v else 1}

    results = [{"id": r["id"], **res[r["id"]],
                "status": "PASS" if res[r["id"]]["failed_count"] == 0 else "FAIL"}
               for r in rules]
    out = {"parquet": parquet, "rows": n_rows, "results": results,
           "failed_rules": sum(x["status"] == "FAIL" for x in results),
           "passed_rules": sum(x["status"] == "PASS" for x in results)}

    os.makedirs(output_dir, exist_ok=True)
    out_path = os.path.join(output_dir, "results.json")
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2)
    return out, out_path

In [2]:
if __name__ == "__main__":
    # In a notebook these default to the current folder. As a .py:
    #   python RunRulesMin.py --parquet_dir ... --rules_dir ... --output_dir ...
    p = argparse.ArgumentParser()
    p.add_argument("--parquet_dir", default=os.getcwd())
    p.add_argument("--rules_dir",   default=os.getcwd())
    p.add_argument("--output_dir",  default=os.getcwd())
    args, _ = p.parse_known_args()   # ignore Jupyter's -f kernel argument

    out, out_path = main(args.parquet_dir, args.rules_dir, args.output_dir)

    print("%s rows | %d failed / %d passed" % (format(out["rows"], ","), out["failed_rules"], out["passed_rules"]))
    print("written: %s\n" % out_path)
    print("%-32s%14s  status" % ("rule", "failed_count"))
    print("-" * 56)
    for r in out["results"]:
        print("%-32s%14s  %s" % (r["id"], format(r["failed_count"], ","), r["status"]))

50,000,000 rows | 10 failed / 10 passed
written: C:\Users\Shadab\Personal_Projects\DUCKDB\dq-engine-compare\results.json

rule                              failed_count  status
--------------------------------------------------------
COMP_IS_NULL                                 0  PASS
COMP_NOT_NULL                        1,049,500  FAIL
STAT_MAX_BETWEEN                             1  FAIL
STAT_MIN_BETWEEN                             0  PASS
STAT_SUM_BETWEEN                             1  FAIL
STAT_MEAN_BETWEEN                            1  FAIL
STAT_MEDIAN_BETWEEN                          0  PASS
STAT_STDDEV_BETWEEN                          0  PASS
STAT_MODE_IN_SET                             0  PASS
VAL_BETWEEN                            281,500  FAIL
VAL_LENGTH_BETWEEN                     594,000  FAIL
VAL_IN_SET                             529,000  FAIL
VAL_NOT_IN_SET                               0  PASS
VAL_REGEX_MATCH                      1,170,500  FAIL
VAL_REGEX_NOT_MATCH     